# Pre-Course Readiness Check

**Artifact:** `PRECOURSE-READINESS`  
**Purpose:** Confirm that the CPU notebook environment can run the deterministic operations used on Days 1 and 2.  
**Estimated duration:** 10-15 minutes  
**Prerequisites:** Basic Python expressions, function calls, and array indexing.

This is a setup check, not a primary lab. It downloads nothing, trains no neural network, and contains no graded challenge.

## Setup Paths

- **Local:** Follow `courseware/shared/environment.md`, select the named kernel, and restart it before this check.
- **Google Colab:** Choose a CPU runtime. Run the version cell first; install packages only if an import fails. Hosted Colab remains a separate validation target.
- **Compute:** CPU only; expected compute time is under 10 seconds.

For the release check, use **Restart Kernel and Run All**. Every success message should appear in order.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression

SEED = 17
rng = np.random.default_rng(SEED)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"PyTorch: {torch.__version__}")
print("PASS: core imports succeeded.")

### What the version output means

The locally validated versions are Python 3.11.8, NumPy 2.4.6, matplotlib 3.11.1, scikit-learn 1.9.0, and PyTorch 2.6.0. A different version is not an automatic failure, but Lab Tester must validate the affected evidence bands before claiming it is compatible. An exact PyTorch 2.11 runtime and hosted Colab have not been execution-validated.

## Check 1: Deterministic Arrays and Shapes

The course treats shapes as executable predictions. This check creates a batch with 5 examples and 4 features, then applies a dense transformation with 3 outputs.

In [ ]:
X_check = rng.normal(size=(5, 4))
W_check = rng.normal(size=(4, 3))
b_check = np.zeros(3)
Z_check = X_check @ W_check + b_check

assert X_check.shape == (5, 4)
assert Z_check.shape == (5, 3)
assert np.isfinite(Z_check).all()
print("PASS: NumPy matrix multiplication and bias broadcasting produced shape (5, 3).")

## Check 2: Plot Rendering

The plot should show two labeled lines with different line styles and point markers. The distinction does not rely on color alone.

In [ ]:
x_plot = np.linspace(-2.0, 2.0, 9)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x_plot, x_plot, marker="o", linestyle="-", label="linear: y = x")
ax.plot(x_plot, x_plot**2, marker="s", linestyle="--", label="nonlinear: y = x^2")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set(xlabel="Input x", ylabel="Output y", title="Readiness plot: linear and nonlinear relationships")
ax.legend()
ax.grid(alpha=0.25)
plt.show()
print("PASS: matplotlib rendered the readiness plot.")

## Check 3: Local Dataset and Estimator

This check generates data in memory. It uses explicit estimator settings so an upstream default change is less likely to alter the result. The expected training accuracy is at least 0.95; this confirms the setup, not model quality.

In [ ]:
X_ready, y_ready = make_blobs(
    n_samples=80,
    centers=[(-1.5, -1.0), (1.5, 1.0)],
    cluster_std=0.35,
    random_state=SEED,
)
ready_model = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=500,
    random_state=SEED,
)
ready_model.fit(X_ready, y_ready)
ready_accuracy = ready_model.score(X_ready, y_ready)

assert X_ready.shape == (80, 2)
assert 0.95 <= ready_accuracy <= 1.0
print(f"PASS: scikit-learn generated local data and fit the smoke-test model (accuracy={ready_accuracy:.3f}).")

## Check 4: PyTorch CPU and Device Visibility

The required course path is CPU. The final code cell performs one small tensor operation explicitly on CPU, then reports an optional available accelerator without selecting it for any lab.

## Final Readiness Check

You are ready for the required Day 1 and Day 2 CPU path when every `PASS` message appears and the plot is visible. The next notebooks will ask you to predict before running, modify focused `TODO` sections, inspect plots, and explain evidence.

In [ ]:
cpu_inputs = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    dtype=torch.float32,
    device="cpu",
)
cpu_weights = torch.tensor([[0.5], [1.5]], dtype=torch.float32, device="cpu")
cpu_outputs = cpu_inputs @ cpu_weights

assert cpu_outputs.shape == (2, 1)
assert cpu_outputs.device.type == "cpu"
assert bool(torch.isfinite(cpu_outputs).all())

mps_available = bool(
    getattr(torch.backends, "mps", None)
    and torch.backends.mps.is_available()
)
optional_device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if mps_available
    else "cpu"
)
print(f"Optional available device: {optional_device}")
print("PASS: PyTorch completed the required tensor check on CPU.")

checks = {
    "array_shape": Z_check.shape == (5, 3),
    "finite_values": bool(np.isfinite(Z_check).all()),
    "local_estimator": ready_accuracy >= 0.95,
    "torch_cpu": cpu_outputs.shape == (2, 1) and cpu_outputs.device.type == "cpu",
}
assert all(checks.values()), checks
print("READY: all deterministic pre-course checks passed on CPU.")

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| Import fails | Wrong kernel or missing package | Select the course kernel or use the install command in `environment.md`, then restart |
| Shape assertion fails | A setup cell was edited or run out of order | Restart and run all without changing supplied values |
| Plot is blank | Backend/kernel state is stale | Restart, run imports, and rerun the plotting cell |
| Accuracy is outside the band | Seed or estimator settings changed | Restore the supplied setup and rerun from a clean kernel |
| PyTorch import or CPU check fails | PyTorch is missing from the selected kernel or tensors were moved to another device | Follow `environment.md`, select the course kernel, restart, and keep this check on CPU |
| Optional device reports `cpu` | No supported accelerator is available | Continue on CPU; it is the required path |

### Lab Tester instructions

Run from a clean local CPU kernel. Confirm the ordered pass messages, visible labeled plot, observed versions, runtime, and absence of network access after package installation. When hosted validation is available, repeat in a fresh Colab CPU runtime and record it separately; do not infer hosted readiness from the local run. This notebook has no learner-completed path and should not be evaluated as a primary lab.